In [31]:
import os, csv, json, time
from pathlib import Path
#INPUT ARGUMENTS
OUT_DIR = Path("ctspinopelvic1k_sample_normal")
N_SAMPLES = 10
SEED = 20260911
REVISION="main"
METHODS = ['ground_truth','nvsegment','totalsegmentator']
DEVICE = 'auto' # auto -> Apple GPU (MPS); a case that fails on MPS is retried on CPU
WORKERS = max(1, min(8, (os.cpu_count() or 4) - 2))  # meshing runs this many cases in parallel (~1-3 GB RAM each)
TET_MODE = "fast"                         # "fast" | "relocate" | "netgen" (all tets pass, ~25x slower) | "none"
WALKTHROUGH_CASE = None                   # volume_id for sections 6 and 8; None = first selected case
WALKTHROUGH_BONE = "vertebra_L3"          # any name from spine_mesh_pipeline/labels.py EXPORT_LABELS

In [32]:
#Check whether we are running the code on NVIDIA/CUDA based computer or mac
import platform
try:
    import torch
    gpu = f"MPS available: {torch.backends.mps.is_available()}, CUDA: {torch.cuda.is_available()}, torch {torch.__version__}"
except ImportError:
    gpu = "torch not installed - section 4 will fail (see setup)"
print(platform.machine(), "| Python", platform.python_version(), "|", os.cpu_count(), "CPU cores |", gpu, "| meshing workers:", WORKERS)

arm64 | Python 3.13.15 | 18 CPU cores | MPS available: True, CUDA: False, torch 2.14.0 | meshing workers: 8


**1. Select the records** \
This code aims to extract out from the manifest json file all the records that have fused spine-pelvic scans with no LSTV, L6, T13 and surgical hardware


In [33]:
import pandas as pd
import csv
import json
import random
from pathlib import Path
import nibabel as nib
import numpy as np
from huggingface_hub import hf_hub_download

In [34]:
#Labels and repository to be used
REPO = "OpenSpineConsortium/CTSpinoPelvic1K"
excluded_list ={
    25: "vertebra_L6_LSTV",
    28: "vertebra_T13",
    46: "rib_left_13",
    59: "rib_right_13",
    60: "rib_left_lumbar",
    61: "rib_right_lumbar",
    62: "hardware_unspecified",
    63: "hardware_cage",
    64: "hardware_screw_rod",
    65: "hardware_plate",
    66: "hardware_arthroplasty",
    67: "hardware_si_screw",
    68: "hardware_osteosynthesis",}

LABELS = {
    **{i:f"vertebra_C{i}" for i in range(1, 8)},#C1 to C7
    **{i:f"vertebra_T{i-7}" for i in range(8, 20)}, #T1 to T12
    **{i:f"vertebra_L{i-19}" for i in range(20, 25)},
    26:"sacrum",
    29:"vertebra_S1",
    30:"hip_left",
    31:"hip_right"
}
MIN_SLICE_THICKNESS = 1
MAX_SLICE_THICKNESS = 1.5


In [35]:
def fetch(relative_path,local_root, revision):
    local = Path(local_root)/relative_path
    if local.exists():
        return local
    return Path(hf_hub_download(repo_id=REPO,repo_type="dataset",revision=revision,filename=relative_path,local_dir=str(local_root),))

In [36]:
# if (OUT / "selected_manifest.csv").exists():
#     cases = load_cases(OUT_DIR)[:N_CASES]
# else:
out = OUT_DIR.resolve()
out.mkdir(parents=True, exist_ok=True)
src = out/"source_nifti"
# display(pd.DataFrame(cases))
# display(pd.read_csv(OUT / "eligibility_audit.csv")["decision"].value_counts().rename("screened candidates"))

In [37]:
manifest_file = fetch('manifest.json',src,REVISION)
records = json.loads(manifest_file.read_text(encoding="utf-8"))
records

[{'volume_id': '0007',
  'token': '5',
  'side': 'fused',
  'position': 'prone',
  'age': 61,
  'age_band': '60-69',
  'sex': 'male',
  'patient_weight': None,
  'convolution_kernel': 'B30f',
  'manufacturer': 'SIEMENS',
  'manufacturer_model': 'Sensation 16',
  'slice_thickness': 1.0,
  'kvp': 120.0,
  'config': 'fused',
  'match_type': 'fused',
  'prov_spine': 'manual',
  'prov_pelvis': 'manual',
  'review_priority': None,
  'lstv_label': 'normal',
  'lstv_class': 0,
  'lstv_pelvic': 'NORMAL',
  'lstv_vertebral': 'NORMAL',
  'castellvi_type': None,
  'castellvi_agreement': None,
  'lstv_agreement': True,
  'lstv_confusion_zone': False,
  'has_l6': False,
  'n_lumbar_labels': 5,
  'has_anchor': True,
  'alignment_ok': True,
  'ct_resampled_to_mask': True,
  'partial_annotation': False,
  'n_voxels_ignore': 0,
  'n_voxels_fg': 0,
  'n_voxels_bg': 101141699,
  'spine_series_uid': '1.3.6.1.4.1.9328.50.4.4716',
  'pelvic_series_uid': '1.3.6.1.4.1.9328.50.4.4716',
  'spine_bone_pct': 64.6,

In [38]:
def is_false(value):
    return str(value).lower().strip() in {"false", "0", "no", "n"}

def eligible_metadata(record):
    #reject any non-fused, transitional, L6 or hardware record
    return(
        str(record.get("config", "")).strip().lower() == "fused"
        and str(record.get("lstv_class", "")).strip() in {"0", "0.0"}
        and is_false(record.get("lstv_confusion_zone", False))
        and is_false(record.get("has_l6", False))
        and is_false(record.get("hardware_labelled", False))
        and str(record.get("position")).strip().lower() in {"prone", "supine"}
        and MIN_SLICE_THICKNESS <= (record.get("slice_thickness")) <= MAX_SLICE_THICKNESS
        and is_false(record.get("has_lumbar_rib",False))
        and record.get("ct_file") and record.get("label_file")
    )

In [39]:
#scan through the records and only keep those that do not have the exclusion criteria
candidates = [r for r in records if eligible_metadata(r)]
sum(1 for item in candidates if isinstance(item, dict)) #Verifies we have exactly 334 records after filtering
random.Random(SEED).shuffle(candidates)

In [43]:
def voxel_exclusion(label_path):
    labels = np.asanyarray(nib.load(str(label_path)).dataobj)#nib used to load neuroimaging (NIFTI) files, dataobj returns an array proxy object (memory-saving feature)
    ids = set(np.unique(labels).astype(int).tolist())
    return {label_id: excluded_list[label_id] for label_id in excluded_list if label_id in ids}




In [44]:
selected, audit = [],[]
#Inspect labels before CT Download
for record in candidates:
    volume_id = str(record.get("volume_id", record.get("token","unknown")))
    try:
        label_path = fetch(record['label_file'], src, REVISION)
        excluded = voxel_exclusion(label_path)
        decision = "accepted" if not excluded else "rejected_voxels_"+";".join(excluded.values())
    except Exception as e:
        label_path, excluded = None, {}
        decision = f"rejected_label_check_error_{type(e).__name__}: {e}"
    audit.append({
            "volume_id": volume_id, "ct_file": record.get("ct_file", ""), "label_file": record.get("label_file", ""),
            "lstv_class": record.get("lstv_class", ""), "castellvi_type": record.get("castellvi_type", ""),
            "hardware_labelled": record.get("hardware_labelled", ""),
            "forbidden_ids_found": ";".join(map(str, excluded)),
            "forbidden_structures_found": ";".join(excluded.values()), "decision": decision,
        })
    if decision == "accepted":
        selected.append((record,label_path))
    if len(selected) == N_SAMPLES:
        break

In [45]:
with (out / "eligibility_audit.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(audit[0]) if audit else [])
    writer.writeheader(); writer.writerows(audit)
if len(selected) < N_SAMPLES:
    raise SystemExit(f"Only {len(selected)} candidates passed all exclusions; requested {N_SAMPLES}.")

In [49]:
selected

[({'volume_id': '0264',
   'token': '186',
   'side': 'fused',
   'position': 'supine',
   'age': 64,
   'age_band': '60-69',
   'sex': 'male',
   'patient_weight': None,
   'convolution_kernel': 'STANDARD',
   'manufacturer': 'GE MEDICAL SYSTEMS',
   'manufacturer_model': 'LightSpeed16',
   'slice_thickness': 1.25,
   'kvp': 120.0,
   'config': 'fused',
   'match_type': 'fused',
   'prov_spine': 'manual',
   'prov_pelvis': 'manual',
   'review_priority': None,
   'lstv_label': 'normal',
   'lstv_class': 0,
   'lstv_pelvic': 'NORMAL',
   'lstv_vertebral': 'NORMAL',
   'castellvi_type': None,
   'castellvi_agreement': None,
   'lstv_agreement': True,
   'lstv_confusion_zone': False,
   'has_l6': False,
   'n_lumbar_labels': 5,
   'has_anchor': True,
   'alignment_ok': True,
   'ct_resampled_to_mask': True,
   'partial_annotation': False,
   'n_voxels_ignore': 0,
   'n_voxels_fg': 0,
   'n_voxels_bg': 139280803,
   'spine_series_uid': '1.3.6.1.4.1.9328.50.4.211195',
   'pelvic_series_uid

In [50]:
result_fields = ["sample_index", "volume_id", "ct_file", "label_file", "status", "exported_structures"]
with (out / "selected_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=result_fields)
    writer.writeheader()
    for index, (record, label_path) in enumerate(selected, 1):
        volume_id = str(record.get("volume_id", record.get("token", f"case_{index:03d}")))
        row = {"sample_index": index, "volume_id": volume_id, "ct_file": record["ct_file"], "label_file": record["label_file"], "status": "", "exported_structures": ""}
        writer.writerow(row)

In [52]:
(out / "run_info.json").write_text(json.dumps({
        "repo": REPO, "revision": REVISION, "sample_size": N_SAMPLES, "seed": SEED,
        "required_config": "fused",
        "metadata_exclusions": ["lstv_class != 0", "has_l6", "lstv_confusion_zone", "castellvi_type", "lstv_phenotype", "hardware_labelled"],
        "voxel_exclusions": excluded_list,
}, indent=2), encoding="utf-8")

726

In [53]:
"""Rows of the selection manifest: sample_index, volume_id, ct_file, label_file."""
path = Path(out) / "selected_manifest.csv"
if not path.exists():
    raise SystemExit(f"{path} not found - run `python run_pipeline.py select --out {out}` first.")
with path.open(newline="", encoding="utf-8") as f:
    cases = [{"sample_index": int(r["sample_index"]), "volume_id": r["volume_id"],
             "ct_file": r["ct_file"], "label_file": r["label_file"]}
             for r in csv.DictReader(f)
            ]

In [54]:
cases

[{'sample_index': 1,
  'volume_id': '0264',
  'ct_file': 'ct/0264_ct.nii.gz',
  'label_file': 'labels/0264_label.nii.gz'},
 {'sample_index': 2,
  'volume_id': '0535',
  'ct_file': 'ct/0535_ct.nii.gz',
  'label_file': 'labels/0535_label.nii.gz'},
 {'sample_index': 3,
  'volume_id': '0827',
  'ct_file': 'ct/0827_ct.nii.gz',
  'label_file': 'labels/0827_label.nii.gz'},
 {'sample_index': 4,
  'volume_id': '0368',
  'ct_file': 'ct/0368_ct.nii.gz',
  'label_file': 'labels/0368_label.nii.gz'},
 {'sample_index': 5,
  'volume_id': '0544',
  'ct_file': 'ct/0544_ct.nii.gz',
  'label_file': 'labels/0544_label.nii.gz'},
 {'sample_index': 6,
  'volume_id': '0613',
  'ct_file': 'ct/0613_ct.nii.gz',
  'label_file': 'labels/0613_label.nii.gz'},
 {'sample_index': 7,
  'volume_id': '1078',
  'ct_file': 'ct/1078_ct.nii.gz',
  'label_file': 'labels/1078_label.nii.gz'},
 {'sample_index': 8,
  'volume_id': '0165',
  'ct_file': 'ct/0165_ct.nii.gz',
  'label_file': 'labels/0165_label.nii.gz'},
 {'sample_index'

In [55]:
import pandas as pd
display(pd.DataFrame(cases))

,sample_index,volume_id,ct_file,label_file
0,1,0264,ct/0264_ct.nii.gz,labels/0264_label.nii.gz
1,2,0535,ct/0535_ct.nii.gz,labels/0535_label.nii.gz
2,3,0827,ct/0827_ct.nii.gz,labels/0827_label.nii.gz
3,4,0368,ct/0368_ct.nii.gz,labels/0368_label.nii.gz
4,5,0544,ct/0544_ct.nii.gz,labels/0544_label.nii.gz
5,6,0613,ct/0613_ct.nii.gz,labels/0613_label.nii.gz
6,7,1078,ct/1078_ct.nii.gz,labels/1078_label.nii.gz
7,8,0165,ct/0165_ct.nii.gz,labels/0165_label.nii.gz
8,9,0438,ct/0438_ct.nii.gz,labels/0438_label.nii.gz
9,10,0329,ct/0329_ct.nii.gz,labels/0329_label.nii.gz


In [57]:
display(pd.read_csv(out/"eligibility_audit.csv")['decision'].value_counts().rename("screened candidates"))

decision
accepted    10
Name: screened candidates, dtype: int64

**2. Download the labels and CT data**

In [60]:
for case in cases:
    for key in ("label_file","ct_file"):
        print(case[key])
        path = fetch(case[key],src, REVISION)
    print(f"{case['volume_id']}: CT {path.stat().st_size / 1e6:.0f} MB")

{'sample_index': 1, 'volume_id': '0264', 'ct_file': 'ct/0264_ct.nii.gz', 'label_file': 'labels/0264_label.nii.gz'}
labels/0264_label.nii.gz
{'sample_index': 1, 'volume_id': '0264', 'ct_file': 'ct/0264_ct.nii.gz', 'label_file': 'labels/0264_label.nii.gz'}
ct/0264_ct.nii.gz
{'sample_index': 2, 'volume_id': '0535', 'ct_file': 'ct/0535_ct.nii.gz', 'label_file': 'labels/0535_label.nii.gz'}
labels/0535_label.nii.gz
{'sample_index': 2, 'volume_id': '0535', 'ct_file': 'ct/0535_ct.nii.gz', 'label_file': 'labels/0535_label.nii.gz'}
ct/0535_ct.nii.gz
{'sample_index': 3, 'volume_id': '0827', 'ct_file': 'ct/0827_ct.nii.gz', 'label_file': 'labels/0827_label.nii.gz'}
labels/0827_label.nii.gz
{'sample_index': 3, 'volume_id': '0827', 'ct_file': 'ct/0827_ct.nii.gz', 'label_file': 'labels/0827_label.nii.gz'}
ct/0827_ct.nii.gz
{'sample_index': 4, 'volume_id': '0368', 'ct_file': 'ct/0368_ct.nii.gz', 'label_file': 'labels/0368_label.nii.gz'}
labels/0368_label.nii.gz
{'sample_index': 4, 'volume_id': '0368', 

**3. Look at the ground truth**
Mid-sagittal (spine) and coronal (pelvis) slices with the labels that will be meshed. Label 30 (`hip_left`) sits on the patient's right in these scans' orientation; the models use the same naming, so comparisons stay consistent.